# Interactive Pupil Function -> PSF

Drag sliders to change pupil parameters. Three panels update live:

- **Left**: Pupil P(kx, ky)
- **Middle**: Focal plane |FT{P}|^2
- **Right**: Beam propagation xz (angular spectrum)

xz uses: `E(x,z) = iFFT{ P * exp(i*2*pi*kz*z) }` where `kz = sqrt(1 - kx^2 - ky^2)`

This is exactly what BioBeam `psf_debye.cl` does.

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


In [ ]:
N  = 256
NZ = 100
Z_HALF = 15

ks   = np.linspace(-1.0, 1.0, N)
KX, KY = np.meshgrid(ks, ks)
KR  = np.sqrt(KX**2 + KY**2)
PHI = np.arctan2(KY, KX)
KZ  = np.sqrt(np.maximum(1.0 - KR**2, 0.0))
zs  = np.linspace(-Z_HALF, Z_HALF, NZ)

BEAM_TYPES = ['DSLM (Gaussian)', 'Bessel (ring)', 'SPIM (cylindrical)', 'Lattice LLS']

def make_pupil(beam_type, NA, NA1, NA2, kpoints, sigma_phi, line_hw):
    if beam_type == 'DSLM (Gaussian)':
        return (KR <= NA).astype(complex)
    elif beam_type == 'Bessel (ring)':
        return ((KR >= NA1) & (KR <= NA2)).astype(complex)
    elif beam_type == 'SPIM (cylindrical)':
        return ((np.abs(KY) < line_hw) & (KR <= NA)).astype(complex)
    elif beam_type == 'Lattice LLS':
        # --- 匹配 BioBeam focus_field_lattice 的实现 ---
        # BioBeam 逻辑 (psf_lattice.cl):
        #   1. kpoints 个多边形顶点放在半径 k_center 的圆上
        #   2. 每个顶点沿 ky 方向做 Gaussian 展宽 (sigma_phi)
        #   3. 环形掩模 (NA1<=KR<=NA2) 截断
        # 物理意义：ky 展宽控制 y 方向的光片厚度 vs 焦深权衡
        k_center = (NA1 + NA2) / 2.0
        ring = (KR >= NA1) & (KR <= NA2)
        # 多边形顶点坐标 (matching BioBeam _poly_points)
        ts  = np.pi * (0.5 + 2.0 / kpoints * np.arange(kpoints))
        kxs = k_center * np.cos(ts)
        kys = k_center * np.sin(ts)
        # kx 方向极窄 (delta-like)，ky 方向用 sigma_phi 展宽
        sigma_kx = k_center * 0.04
        sigma_ky = max(sigma_phi, 1e-4)
        amp = np.zeros((N, N))
        for kx_i, ky_i in zip(kxs, kys):
            amp += (np.exp(-(KX - kx_i)**2 / (2 * sigma_kx**2)) *
                    np.exp(-(KY - ky_i)**2 / (2 * sigma_ky**2)))
        amp *= ring
        if amp.max() > 0:
            amp /= amp.max()
        return amp.astype(complex)

def focal_psf(P):
    E = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(P)))
    return np.abs(E)**2

def xz_profile(P):
    xz = np.zeros((NZ, N), dtype=np.float32)
    for iz, z in enumerate(zs):
        Ep = P * np.exp(1j * 2 * np.pi * KZ * z)
        E  = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(Ep)))
        xz[iz] = np.abs(E[N//2, :])**2
    return xz

def log_norm(img, decades=3):
    eps = img.max() * 1e-10 + 1e-30
    lo  = np.log10(img.max() + eps) - decades
    return np.clip(np.log10(img + eps), lo, None)

print('Setup done')


In [ ]:
w_type = widgets.Dropdown(
    options=BEAM_TYPES, value='DSLM (Gaussian)', description='Beam:',
    style={'description_width': '60px'}, layout=widgets.Layout(width='300px'))
w_NA  = widgets.FloatSlider(min=0.05, max=0.95, step=0.05, value=0.30,
    description='NA', continuous_update=True,
    style={'description_width': '70px'}, layout=widgets.Layout(width='340px'))
w_NA1 = widgets.FloatSlider(min=0.05, max=0.85, step=0.05, value=0.42,
    description='NA1 inner', continuous_update=True,
    style={'description_width': '70px'}, layout=widgets.Layout(width='340px'))
w_NA2 = widgets.FloatSlider(min=0.10, max=0.95, step=0.05, value=0.55,
    description='NA2 outer', continuous_update=True,
    style={'description_width': '70px'}, layout=widgets.Layout(width='340px'))
w_kpts = widgets.IntSlider(min=2, max=10, step=1, value=6,
    description='kpoints', continuous_update=True,
    style={'description_width': '70px'}, layout=widgets.Layout(width='340px'))
w_sigma = widgets.FloatSlider(min=0.02, max=0.60, step=0.02, value=0.15,
    description='sigma_phi', continuous_update=True,
    style={'description_width': '70px'}, layout=widgets.Layout(width='340px'))
w_linehw = widgets.FloatSlider(min=0.005, max=0.10, step=0.005, value=0.015,
    description='line_width', continuous_update=True,
    style={'description_width': '70px'}, layout=widgets.Layout(width='340px'))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('Pupil Function and PSF', fontsize=12)
for ax, t in zip(axes, ['Pupil P(kx,ky)', 'Focal plane |FT{P}|^2', 'Propagation xz']):
    ax.set_title(t, fontsize=10)
    ax.axis('off')

dummy = np.zeros((N, N))
im_p = axes[0].imshow(dummy, cmap='Reds',   origin='lower', vmin=0, vmax=1)
im_f = axes[1].imshow(dummy, cmap='inferno', origin='lower')
im_x = axes[2].imshow(dummy, cmap='inferno', origin='lower', aspect='auto')
theta = np.linspace(0, 2*np.pi, 300)
r_px  = N//2 - 2
axes[0].plot(N//2 + r_px*np.cos(theta), N//2 + r_px*np.sin(theta), 'k--', lw=0.8, alpha=0.6)
CROP = 80

def refresh(_=None):
    P = make_pupil(w_type.value, w_NA.value, w_NA1.value, w_NA2.value,
                   w_kpts.value, w_sigma.value, w_linehw.value)
    im_p.set_data(np.abs(P))
    c = N//2
    xy  = focal_psf(P)[c-CROP:c+CROP, c-CROP:c+CROP]
    lxy = log_norm(xy)
    im_f.set_data(lxy); im_f.set_clim(lxy.min(), lxy.max())
    xz  = xz_profile(P)[:, c-CROP:c+CROP]
    lxz = log_norm(xz)
    im_x.set_data(lxz); im_x.set_clim(lxz.min(), lxz.max())
    fig.canvas.draw_idle()

for w in [w_type, w_NA, w_NA1, w_NA2, w_kpts, w_sigma, w_linehw]:
    w.observe(refresh, names='value')

panel = widgets.VBox([
    w_type,
    widgets.HBox([
        widgets.VBox([widgets.HTML('<b>DSLM / SPIM</b>'),   w_NA, w_linehw]),
        widgets.VBox([widgets.HTML('<b>Bessel / Lattice</b>'), w_NA1, w_NA2]),
        widgets.VBox([widgets.HTML('<b>Lattice only</b>'),   w_kpts, w_sigma]),
    ])
])
display(panel)
refresh()


## Suggested Experiments

**Exp 1 - DSLM: effect of NA**
Drag NA 0.1->0.9. Larger NA = tighter focus (xy), shorter depth of field (xz).

**Exp 2 - Bessel: ring width**
Fix NA1=0.42, drag NA2. Thinner ring = longer non-diffracting propagation, more side lobes.

**Exp 3 - Lattice: kpoints**
Change kpoints 2->4->6. Watch xy pattern change (square vs hex lattice).

**Exp 4 - Lattice: sigma_phi**
Fix kpoints=6, drag sigma_phi 0.02->0.50. Larger sigma = thinner sheet, shorter DOF.
This is the core trade-off in Chen et al. 2014.

**Exp 5 - SPIM vs DSLM**
Same NA, switch beam type. SPIM xz = line focus (sheet directly). DSLM xz = point focus (needs scanning).

---
## Comparison: BioBeam vs `llspy-slm` ([tlambert03/llspy-slm](https://github.com/tlambert03/llspy-slm))

| Aspect | BioBeam (this notebook) | llspy-slm |
|--------|------------------------|-----------|
| **Lattice pupil model** | Gaussian blobs at polygon vertices on ring (σ_φ tunable) | Exact hexagonal plane-wave waveset → real-space pattern → binary 0/π SLM phase → FFT → annular mask |
| **Widefield PSF** | Ideal Airy disc — no aberrations (simple disk aperture) | Gibson-Lanni: Fourier-Bessel expansion of OPD (coverslip RI, immersion RI, particle depth) |
| **Output** | Continuous complex pupil → angular spectrum propagation | Binary SLM bitmap for hardware; optional intensity at sample |

### Lattice LLS approach

**BioBeam** places `kpoints` Gaussian blobs at polygon vertices on the annular ring.  
`sigma_phi` controls the ky spread → trade-off between sheet thickness and depth of field.

**llspy-slm `hex_lattice`**:
1. Precomputed `Hex_Fund_MaxComp` waveset: 6 exact plane-wave k-directions
2. Superpose plane waves in real space → ideal 2D hexagonal lattice
3. Apply Gaussian bounding envelope (width ∝ π / (NA_outer − NA_inner))
4. Binarize to 0/π SLM phase → FFT creates diffraction orders
5. Annular mask selects the desired ring → field at sample

The binarization step is the key hardware-facing difference: llspy-slm generates actual SLM bitmap files.

In [ ]:
# =============================================================================
# llspy-slm Lattice LLS comparison
# Hex_Fund_MaxComp waveset: 6 plane-wave unit-vector k-directions [kx, ky]
# (from slmgen/slm.py, columns 3-4 of the waveset array, normalized)
# =============================================================================

WAVESET_HEX = np.array([
    [ 0.0,        -1.0      ],
    [ 0.8660254,  -0.5      ],
    [ 0.8660254,   0.5      ],
    [ 0.0,         1.0      ],
    [-0.8660254,   0.5      ],
    [-0.8660254,  -0.5      ],
])

# Real-space coordinates conjugate to our k-space (dk = 2/N → dx = N/(2N) = 0.5)
_xr = np.fft.fftshift(np.fft.fftfreq(N, d=2.0/N))  # range [-N/4, N/4), step 0.5
_XR, _YR = np.meshgrid(_xr, _xr)


def make_pupil_llspy_ideal(NA1, NA2, sigma_k=0.012):
    """llspy-slm ideal: near-delta Gaussians at waveset positions + annular mask."""
    k_c  = (NA1 + NA2) / 2.0
    ring = (KR >= NA1) & (KR <= NA2)
    amp  = np.zeros((N, N))
    for kxn, kyn in WAVESET_HEX:
        amp += np.exp(-((KX - k_c*kxn)**2 + (KY - k_c*kyn)**2) / (2*sigma_k**2))
    amp *= ring
    if amp.max() > 0:
        amp /= amp.max()
    return amp.astype(complex)


def make_pupil_llspy_slm(NA1, NA2, fill_factor=0.75, crop=0.15):
    """
    llspy-slm hex_lattice full pipeline:
      ideal k-space pupil → IFFT → Gaussian bounding envelope →
      binarize (0/π SLM phase) → FFT → annular mask
    """
    P_ideal = make_pupil_llspy_ideal(NA1, NA2)

    # IFFT → real-space lattice (coherent sum of 6 plane waves)
    E_real = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(P_ideal)))
    RealE  = np.real(E_real)

    # Gaussian bounding envelope along x (confinement set by annular bandwidth)
    kxdiff    = max(NA2 - NA1, 0.01)
    sigma_env = (np.pi / kxdiff) / (np.sqrt(2.0 * np.log(2.0)) * fill_factor)
    RealE    *= np.exp(-2.0 * (_XR / sigma_env)**2)

    # Normalize, apply crop threshold, binarize to 0/π SLM phase
    RealE /= np.abs(RealE).max() + 1e-30
    RealE[np.abs(RealE) < crop] = 0.0
    eps       = np.finfo(float).eps
    slm_phase = np.sign(RealE + eps) * np.pi/2 + np.pi/2   # 0 or π
    E_slm     = np.exp(1j * slm_phase)

    # FFT back to k-space; apply annular mask
    E_pupil  = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(E_slm)))
    ring     = (KR >= NA1) & (KR <= NA2)
    E_pupil *= ring
    amp      = np.abs(E_pupil)
    if amp.max() > 0:
        amp /= amp.max()
    return amp.astype(complex)


# ---- Compute three pupils (NA matching notebook defaults) ----
NA1_c, NA2_c = 0.42, 0.55
print('Computing pupils…')
P_bb    = make_pupil('Lattice LLS', None, NA1_c, NA2_c, 6, 0.15, None)
P_ll_id = make_pupil_llspy_ideal(NA1_c, NA2_c)
P_ll_sl = make_pupil_llspy_slm(NA1_c, NA2_c)
print('Done. Computing PSFs (xz may take ~30 s)…')

pupils = [P_bb, P_ll_id, P_ll_sl]
titles = [
    'BioBeam\n(Gaussian blobs, σ_φ=0.15)',
    'llspy-slm ideal\n(δ-functions at waveset)',
    'llspy-slm SLM binary\n(0/π phase → FFT → annular mask)',
]
row_labels = ['Pupil P(kx, ky)', 'Focal plane |FT{P}|² (log)', 'xz propagation (log)']

c = N//2; CRP = 60
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
fig.suptitle(f'Lattice LLS — BioBeam vs llspy-slm  (NA_inner={NA1_c}, NA_outer={NA2_c})',
             fontsize=12, y=1.01)

for col, (P, ttl) in enumerate(zip(pupils, titles)):
    axes[0, col].set_title(ttl, fontsize=9)
    for row in range(3):
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(row_labels[row], fontsize=8)

    # Row 0: Pupil
    axes[0, col].imshow(np.abs(P), cmap='Reds', origin='lower', vmin=0, vmax=1)

    # Row 1: Focal PSF
    psf  = focal_psf(P)[c-CRP:c+CRP, c-CRP:c+CRP]
    axes[1, col].imshow(log_norm(psf), cmap='inferno', origin='lower')

    # Row 2: xz
    xz = xz_profile(P)[:, c-CRP:c+CRP]
    axes[2, col].imshow(log_norm(xz), cmap='inferno', origin='lower', aspect='auto')

plt.tight_layout()
plt.savefig('biobeam_vs_llspy_lattice.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved biobeam_vs_llspy_lattice.png')

### Gibson-Lanni PSF (Widefield) — `llspy-slm/slmgen/makepsf.py`

`makePSF()` uses a **Fourier-Bessel series expansion** of the total optical path difference (OPD):

```
W(ρ, z) = (2π/λ) · [OPD_sample + OPD_immersion + OPD_coverslip]
OPD_sample     = d_particle · √(n²_sample − NA²ρ²)
OPD_immersion  = (z + WD)·√(n²_i − NA²ρ²) − WD·√(n²_i0 − NA²ρ²)
OPD_coverslip  = t_g·√(n²_g − NA²ρ²) − t_g0·√(n²_g0 − NA²ρ²)
```

BioBeam's `(KR ≤ NA)` gives a perfect **Airy** PSF — no aberrations at all.  
`makePSF()` gives a **realistic aberrated** PSF when sample RI ≠ immersion RI, or particle is deep in sample.

In [ ]:
from scipy import special as _spc
from math import sqrt as _sqrt

def makePSF_llspy(wavelength=0.525, NA=0.9, nx=129, nz=129, dx=0.05, dz=0.05,
                  RI=1.33, immRI=1.515, csRI=1.515, csthick=170,
                  workingdistance=150, particledistance=0,
                  num_basis=100, num_samples=500):
    """
    Gibson-Lanni scalar PSF — ported from llspy-slm/slmgen/makepsf.py
    Returns PSF_rz of shape (nz, max_radius), values in [0, 1].
    Focal plane = row nz//2.  Radial step = dx μm.
    """
    ni = ni0 = immRI
    ng = ng0 = csRI
    tg = tg0 = csthick
    min_wl = 0.436
    sf  = NA * (3*np.arange(1, num_basis+1) - 2) * min_wl / wavelength
    x0  = (nx - 1) / 2
    mr  = round(_sqrt(2) * (nx - x0)) + 1
    r   = dx * np.arange(0, mr)
    a   = min(NA, RI, ni, ni0, ng, ng0) / NA
    rho = np.linspace(0, a, num_samples)
    z   = dz * np.arange(-nz/2, nz/2) + dz/2
    NArho2 = NA**2 * rho**2
    OPDs = particledistance * np.sqrt(np.maximum(RI**2  - NArho2, 0))
    OPDi = ((z[:,None] + workingdistance) * np.sqrt(np.maximum(ni**2  - NArho2, 0))
            - workingdistance             * np.sqrt(np.maximum(ni0**2 - NArho2, 0)))
    OPDg = (tg  * np.sqrt(np.maximum(ng**2  - NArho2, 0))
            - tg0 * np.sqrt(np.maximum(ng0**2 - NArho2, 0)))
    W     = 2*np.pi/wavelength * (OPDs + OPDi + OPDg)
    phase = np.cos(W) + 1j*np.sin(W)
    J     = _spc.jv(0, sf[:,None] * rho)
    C, *_ = np.linalg.lstsq(J.T, phase.T, rcond=None)
    b     = 2*np.pi * r[:,None] * NA / wavelength
    denom = sf*sf - b*b
    R     = (sf * _spc.jv(1, sf*a) * _spc.jv(0, b*a) * a
             - b * _spc.jv(0, sf*a) * _spc.jv(1, b*a) * a) / denom
    PSF_rz = (np.abs(R.dot(C))**2).T
    PSF_rz /= PSF_rz.max()
    return PSF_rz   # (nz, mr)


# --- Parameters ---
kw_base = dict(wavelength=0.488, NA=0.8, nx=129, nz=129, dx=0.05, dz=0.05,
               RI=1.33, immRI=1.515, csRI=1.515, csthick=170, workingdistance=150)

print('Computing Gibson-Lanni PSF (no aberration)…')
GL_ideal = makePSF_llspy(**kw_base, particledistance=0)
print('Computing Gibson-Lanni PSF (15 μm into water sample, glass immersion)…')
GL_aber  = makePSF_llspy(**kw_base, particledistance=15)

# BioBeam ideal Airy (focal plane 2D crop)
P_airy   = make_pupil('DSLM (Gaussian)', kw_base['NA'], None, None, None, None, None)
psf_bb2d = focal_psf(P_airy)[N//2-60:N//2+60, N//2-60:N//2+60]

nz2 = GL_ideal.shape[0] // 2
nr  = GL_ideal.shape[1]
r_um = np.arange(nr) * kw_base['dx']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle(f"Widefield PSF: BioBeam (Debye, ideal) vs llspy-slm Gibson-Lanni"
             f"  NA={kw_base['NA']}  λ=488 nm", fontsize=11)

# Panel 0 — BioBeam focal plane
axes[0].imshow(log_norm(psf_bb2d), cmap='inferno', origin='lower')
axes[0].set_title('BioBeam focal plane (log)\nIdeal Airy — no aberrations', fontsize=9)
axes[0].axis('off')

# Panel 1 — Gibson-Lanni xz, no aberration
axes[1].imshow(np.log10(GL_ideal + 1e-6), cmap='inferno', origin='lower', aspect='auto')
axes[1].set_title('llspy-slm GL xz (log)\nNo aberration (particle at coverslip)', fontsize=9)
axes[1].axis('off')

# Panel 2 — Gibson-Lanni xz, 15 μm depth
axes[2].imshow(np.log10(GL_aber + 1e-6), cmap='inferno', origin='lower', aspect='auto')
axes[2].set_title('llspy-slm GL xz (log)\n15 μm into water (n=1.33), glass obj (n=1.515)', fontsize=9)
axes[2].axis('off')

plt.tight_layout()
plt.savefig('biobeam_vs_llspy_widefield_xz.png', dpi=150, bbox_inches='tight')
plt.show()

# Radial profile overlay
fig2, ax2 = plt.subplots(figsize=(7, 4))
ax2.semilogy(r_um, GL_ideal[nz2] / GL_ideal[nz2, 0], lw=2,
             label='GL focal — no aberration')
ax2.semilogy(r_um, GL_aber[nz2]  / GL_aber[nz2, 0],  lw=2, ls='--',
             label='GL focal — 15 μm depth (water/glass RI mismatch)')
# BioBeam radial: pixel coords (no absolute scale in this notebook)
psf_r_bb = psf_bb2d[60, 60:] / psf_bb2d[60, 60]
ax2.semilogy(np.linspace(0, r_um[-1], len(psf_r_bb)), psf_r_bb, lw=2, ls=':',
             label='BioBeam ideal Airy (scale ≈ matched)')
ax2.set_xlabel('r (μm)'); ax2.set_ylabel('Intensity (log, normalized to peak)')
ax2.set_title(f"Focal-plane radial profiles  NA={kw_base['NA']}  λ=488 nm")
ax2.set_xlim(0, 2.0); ax2.set_ylim(1e-4, 1.5)
ax2.legend(fontsize=8)
fig2.tight_layout()
plt.savefig('biobeam_vs_llspy_widefield_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved biobeam_vs_llspy_widefield_xz.png + biobeam_vs_llspy_widefield_profiles.png')